In [2]:
import pandas as pd
books=pd.read_csv("books_with_categories.csv")

In [4]:
# print(books.head())

In [1]:
print("Everything about the book with id 1:")

Everything about the book with id 1:


In [ ]:
from transformers import pipeline

# Load the model
classifier = pipeline("text-classification", 
                      model="j-hartmann/emotion-english-distilroberta-base",
                      device=-1,
                      top_k=None)

text = "A young widow struggles to keep her family together amidst a shocking secret."
results = classifier(text)
print(results)
# # Sort results to see the top emotion
# top_emotion = max(results[0], key=lambda x: x['score'])
# print(f"Detected Emotion: {top_emotion['label']} ({top_emotion['score']:.2f})")

Device set to use cpu


[[{'label': 'fear', 'score': 0.8481522798538208}, {'label': 'surprise', 'score': 0.07337118685245514}, {'label': 'sadness', 'score': 0.051971543580293655}, {'label': 'neutral', 'score': 0.008473235182464123}, {'label': 'joy', 'score': 0.00835446547716856}, {'label': 'anger', 'score': 0.004974630661308765}, {'label': 'disgust', 'score': 0.004702635109424591}]]


In [9]:
classifier(books["tagged_description"][0].split("."))


[[{'label': 'surprise', 'score': 0.6645126938819885},
  {'label': 'joy', 'score': 0.13457171618938446},
  {'label': 'neutral', 'score': 0.1292700171470642},
  {'label': 'fear', 'score': 0.056731585413217545},
  {'label': 'anger', 'score': 0.01058531366288662},
  {'label': 'sadness', 'score': 0.00275967619381845},
  {'label': 'disgust', 'score': 0.0015689738793298602}],
 [{'label': 'neutral', 'score': 0.4493703842163086},
  {'label': 'disgust', 'score': 0.2735915184020996},
  {'label': 'joy', 'score': 0.10908320546150208},
  {'label': 'sadness', 'score': 0.09362740069627762},
  {'label': 'anger', 'score': 0.04047832638025284},
  {'label': 'surprise', 'score': 0.02697017230093479},
  {'label': 'fear', 'score': 0.006879048887640238}],
 [{'label': 'neutral', 'score': 0.6462150812149048},
  {'label': 'sadness', 'score': 0.24273402988910675},
  {'label': 'disgust', 'score': 0.04342275485396385},
  {'label': 'surprise', 'score': 0.028300579637289047},
  {'label': 'joy', 'score': 0.01421149261

In [12]:
sentences = books["tagged_description"][0].split(".")
predictions = classifier(sentences) 
sentences[3]

' Haunted by his grandfather’s presence, John tells of the rift between his grandfather and his father: the elder, an angry visionary who fought for the abolitionist cause, and his son, an ardent pacifist'

In [13]:
predictions[3]

[{'label': 'fear', 'score': 0.9281681180000305},
 {'label': 'anger', 'score': 0.03219105303287506},
 {'label': 'neutral', 'score': 0.012808670289814472},
 {'label': 'sadness', 'score': 0.008756876923143864},
 {'label': 'surprise', 'score': 0.00859791599214077},
 {'label': 'disgust', 'score': 0.008431825786828995},
 {'label': 'joy', 'score': 0.001045584212988615}]

In [14]:
sorted(predictions[0], key=lambda x: x['label'])

[{'label': 'anger', 'score': 0.01058531366288662},
 {'label': 'disgust', 'score': 0.0015689738793298602},
 {'label': 'fear', 'score': 0.056731585413217545},
 {'label': 'joy', 'score': 0.13457171618938446},
 {'label': 'neutral', 'score': 0.1292700171470642},
 {'label': 'sadness', 'score': 0.00275967619381845},
 {'label': 'surprise', 'score': 0.6645126938819885}]

In [20]:
import numpy as  np
emotion_labels = ["anger", "disgust", "joy","fear", "neutral", "sadness", "surprise"]
isbn=[]
emotion_scores={label:[] for label in emotion_labels}

def calculate_max_emotion_scores(predictions):
    per_emotion_scores = {label: [] for label in emotion_labels}
    for prediction in predictions:
        sorted_predictions = sorted(prediction, key=lambda x: x['label'])
        for  index, label in enumerate(emotion_labels):
            per_emotion_scores[label].append(sorted_predictions[index]['score'])
    return {label: np.max(scores) for label, scores in per_emotion_scores.items()}

In [26]:
from tqdm import tqdm
emotion_labels = ["anger", "disgust", "joy","fear", "neutral", "sadness", "surprise"]
isbn=[]
emotion_scores={label:[] for label in emotion_labels}
for i in tqdm(range(len(books))):
    isbn.append(books["isbn13"][i])
    predictions = classifier(books["tagged_description"][i].split("."))
    predictions=classifier(sentences)
    max_scores = calculate_max_emotion_scores(predictions)
    for label in emotion_labels:
        emotion_scores[label].append(max_scores[label])

100%|██████████| 5197/5197 [42:03<00:00,  2.06it/s]  


In [34]:
emotion_df=pd.DataFrame(emotion_scores)
emotion_df["isbn13"]=isbn

In [35]:
emotion_df

,anger,disgust,joy,fear,neutral,sadness,surprise,isbn13
0,0.064134,0.273592,0.928168,0.932798,0.646215,0.967158,0.664513,9780002005883
1,0.064134,0.273592,0.928168,0.932798,0.646215,0.967158,0.664513,9780002261982
2,0.064134,0.273592,0.928168,0.932798,0.646215,0.967158,0.664513,9780006178736
3,0.064134,0.273592,0.928168,0.932798,0.646215,0.967158,0.664513,9780006280897
4,0.064134,0.273592,0.928168,0.932798,0.646215,0.967158,0.664513,9780006280934
...,...,...,...,...,...,...,...,...
5192,0.064134,0.273592,0.928168,0.932798,0.646215,0.967158,0.664513,9788172235222
5193,0.064134,0.273592,0.928168,0.932798,0.646215,0.967158,0.664513,9788173031014
5194,0.064134,0.273592,0.928168,0.932798,0.646215,0.967158,0.664513,9788179921623
5195,0.064134,0.273592,0.928168,0.932798,0.646215,0.967158,0.664513,9788185300535


In [37]:
books =pd.merge(books, emotion_df, on="isbn13")
# books

In [38]:
books

,isbn13,isbn10,title,authors,categories,thumbnail,published_year,average_rating,num_pages,ratings_count,...,tagged_description,broad_category,books_category,anger,disgust,joy,fear,neutral,sadness,surprise
0,9780002005883,0002005883,Gilead,Marilynne Robinson,Fiction,http://books.google.com/books/content?id=KQZCP...,2004.0,3.85,247.0,361.0,...,9780002005883 | A NOVEL THAT READERS and criti...,Fiction,Fiction,0.064134,0.273592,0.928168,0.932798,0.646215,0.967158,0.664513
1,9780002261982,0002261987,Spider's Web,Charles Osborne;Agatha Christie,Detective and mystery stories,http://books.google.com/books/content?id=gA5GP...,2000.0,3.83,241.0,5164.0,...,9780002261982 | A new 'Christie for Christmas'...,NaN,Fiction,0.064134,0.273592,0.928168,0.932798,0.646215,0.967158,0.664513
2,9780006178736,0006178731,Rage of angels,Sidney Sheldon,Fiction,http://books.google.com/books/content?id=FKo2T...,1993.0,3.93,512.0,29532.0,...,"9780006178736 | A memorable, mesmerizing heroi...",Fiction,Fiction,0.064134,0.273592,0.928168,0.932798,0.646215,0.967158,0.664513
3,9780006280897,0006280897,The Four Loves,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=XhQ5X...,2002.0,4.15,170.0,33684.0,...,9780006280897 | Lewis' work on the nature of l...,NaN,NonFiction,0.064134,0.273592,0.928168,0.932798,0.646215,0.967158,0.664513
4,9780006280934,0006280935,The Problem of Pain,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=Kk-uV...,2002.0,4.09,176.0,37569.0,...,"9780006280934 | ""In The Problem of Pain, C.S. ...",NaN,NonFiction,0.064134,0.273592,0.928168,0.932798,0.646215,0.967158,0.664513
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5192,9788172235222,8172235224,Mistaken Identity,Nayantara Sahgal,Indic fiction (English),http://books.google.com/books/content?id=q-tKP...,2003.0,2.93,324.0,0.0,...,9788172235222 | On A Train Journey Home To Nor...,NaN,Fiction,0.064134,0.273592,0.928168,0.932798,0.646215,0.967158,0.664513
5193,9788173031014,8173031010,Journey to the East,Hermann Hesse,Adventure stories,http://books.google.com/books/content?id=rq6JP...,2002.0,3.70,175.0,24.0,...,9788173031014 | This book tells the tale of a ...,NaN,NonFiction,0.064134,0.273592,0.928168,0.932798,0.646215,0.967158,0.664513
5194,9788179921623,817992162X,The Monk Who Sold His Ferrari: A Fable About F...,Robin Sharma,Health & Fitness,http://books.google.com/books/content?id=c_7mf...,2003.0,3.82,198.0,1568.0,...,9788179921623 | Wisdom to Create a Life of Pas...,NaN,Fiction,0.064134,0.273592,0.928168,0.932798,0.646215,0.967158,0.664513
5195,9788185300535,8185300534,I Am that,Sri Nisargadatta Maharaj;Sudhakar S. Dikshit,Philosophy,http://books.google.com/books/content?id=Fv_JP...,1999.0,4.51,531.0,104.0,...,9788185300535 | This collection of the timeles...,NonFiction,NonFiction,0.064134,0.273592,0.928168,0.932798,0.646215,0.967158,0.664513


In [39]:
books.to_csv("books_with_emotions.csv", index=False)